# Modelo genérico · Regressão Linear com gradiente descendente

**Template reutilizável** para prever um valor numérico contínuo a partir de uma ou mais variáveis, implementando a regressão linear do zero (custo, gradiente e gradiente descendente em lote) e validando o resultado contra a solução de mínimos quadrados do scikit-learn.

| | |
|---|---|
| **Modelo no sistema** | Machine Learning › 2. Aprendizado supervisionado › cartão *Regressão Linear* (`/modelar/aprendizado-supervisionado/#m-linear-regression`) |
| **Origem** | `C1_W2_Linear_Regression.ipynb` (*Machine Learning Specialization*, DeepLearning.AI e Stanford Online; lucro de restaurantes × população da cidade), revisado, corrigido e generalizado |
| **Tipo de problema** | Supervisionado · regressão |
| **Dependências** | `numpy`, `pandas`, `matplotlib`, `scikit-learn` |

### Como usar
1. Edite apenas a célula **1. Configuração** (marcada com `>>> ADAPTAR`).
2. Com `CAMINHO_DADOS = None`, o notebook roda de ponta a ponta com dados sintéticos, para você ver o fluxo antes de trocar pelos seus dados.
3. Execute tudo em ordem (*Run All*). Nenhuma outra célula depende de nomes de colunas.

### O que mudou em relação ao notebook original
- As funções `compute_cost` e `compute_gradient` estavam **definidas duas vezes** na mesma célula (a primeira versão, sem `return`, era sobrescrita): ficou só a versão correta, vetorizada.
- O modelo deixou de ser restrito a **uma variável** (`f(x) = wx + b`): agora aceita *n* features (`f(x) = w·x + b`).
- Entraram **divisão treino/teste**, **padronização** ajustada só no treino e as **métricas do cartão** (MAE, RMSE, R²), além da análise de resíduos. O original só acompanhava o custo no próprio treino.
- Os testes do autograder (`public_tests`) e o `utils.py`, que não acompanham o arquivo, foram trocados por uma **verificação numérica do gradiente** e pela comparação com `LinearRegression`.
- Sem caminhos, colunas, unidades ou valores do dataset original.

## 1. Configuração

In [ ]:
# >>> ADAPTAR: única célula que precisa ser editada para usar outro dataset.

# --- dados ---
CAMINHO_DADOS = None        # ex.: "dados/meu_arquivo.csv"; None = dados sintéticos de demonstração
SEPARADOR = None            # None = detecta (CSV/TXT); ex.: ";" ou "\t"
TEM_CABECALHO = True        # False se a primeira linha já é dado (como o ex1data1.txt original)
NOMES_COLUNAS = None        # nomes quando TEM_CABECALHO = False; sem nomes, viram coluna_0, coluna_1, …
COLUNA_ALVO = "alvo"        # coluna numérica a prever
COLUNAS_FEATURES = None     # lista de colunas explicativas; None = todas as demais

# --- divisão e reprodutibilidade ---
TAMANHO_TESTE = 0.2         # fração para teste; 0 = sem teste (só para datasets minúsculos)
SEMENTE = 42

# --- gradiente descendente ---
TAXA_APRENDIZADO = 0.01     # alpha
NUM_ITERACOES = 1500
PADRONIZAR = True           # recomendado: sem escala comum o gradiente descendente converge mal

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

np.random.seed(SEMENTE)

## 2. Carregamento dos dados

In [ ]:
def carregar_tabela(caminho, separador=None, tem_cabecalho=True, nomes_colunas=None, planilha=0):
    """Lê CSV/TXT/TSV, Excel, Parquet ou JSON e devolve um DataFrame."""
    caminho = Path(caminho)
    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho.resolve()}")
    ext = caminho.suffix.lower()
    cabecalho = 0 if tem_cabecalho else None
    if ext in {".csv", ".txt", ".tsv"}:
        sep = separador if separador is not None else ("\t" if ext == ".tsv" else None)
        df = pd.read_csv(caminho, sep=sep, engine="python", header=cabecalho)
    elif ext in {".xlsx", ".xls"}:
        df = pd.read_excel(caminho, sheet_name=planilha, header=cabecalho)
    elif ext == ".parquet":
        df = pd.read_parquet(caminho)
    elif ext == ".json":
        df = pd.read_json(caminho)
    else:
        raise ValueError(f"Formato não suportado: {ext}")
    if nomes_colunas is not None:
        df.columns = nomes_colunas
    elif not tem_cabecalho:
        df.columns = [f"coluna_{i}" for i in range(df.shape[1])]
    return df


def dados_sinteticos(n=200, semente=SEMENTE):
    """Relação linear com ruído e três features, para demonstração."""
    rng = np.random.default_rng(semente)
    X = rng.normal(size=(n, 3)) * [2.0, 1.0, 5.0] + [10.0, 0.0, 50.0]
    y = 3.0 * X[:, 0] - 2.0 * X[:, 1] + 0.5 * X[:, 2] + 7.0 + rng.normal(scale=2.0, size=n)
    df = pd.DataFrame(X, columns=["feature_1", "feature_2", "feature_3"])
    df["alvo"] = y
    return df


df = (dados_sinteticos() if CAMINHO_DADOS is None
      else carregar_tabela(CAMINHO_DADOS, SEPARADOR, TEM_CABECALHO, NOMES_COLUNAS))

if COLUNA_ALVO not in df.columns:
    raise KeyError(f"COLUNA_ALVO={COLUNA_ALVO!r} não existe. Colunas disponíveis: {list(df.columns)}")
if not pd.api.types.is_numeric_dtype(df[COLUNA_ALVO]):
    raise TypeError("A regressão linear exige alvo numérico.")

features = COLUNAS_FEATURES or [c for c in df.columns if c != COLUNA_ALVO]
df = df.dropna(subset=[COLUNA_ALVO])
X_df, y = df[features], df[COLUNA_ALVO].to_numpy(dtype=float)
print("formato:", df.shape, "| features:", features)

### 2.1 Conhecer os dados
Antes de modelar: tipos, faltantes e escala de cada variável. É o equivalente genérico de imprimir `x_train[:5]` e `x_train.shape` do original.

In [ ]:
display(df.head())
display(df.describe(include="all").T)
faltantes = df.isna().sum()
print("faltantes por coluna:", faltantes[faltantes > 0].to_dict() or "nenhum")

In [ ]:
# Dispersão de cada feature numérica contra o alvo (até 6 painéis).
numericas = X_df.select_dtypes(include="number").columns[:6]
if len(numericas):
    fig, eixos = plt.subplots(1, len(numericas), figsize=(4 * len(numericas), 3.5), squeeze=False)
    for ax, col in zip(eixos[0], numericas):
        ax.scatter(X_df[col], y, s=12, alpha=0.6)
        ax.set_xlabel(col)
        ax.set_ylabel(COLUNA_ALVO)
    fig.suptitle(f"{COLUNA_ALVO} em função de cada feature")
    plt.tight_layout()
    plt.show()

## 3. Preparação
A divisão vem **antes** de qualquer ajuste: imputação, one-hot e padronização aprendem só com o treino e são apenas aplicadas ao teste. No one-hot, a primeira categoria de cada variável é descartada: com o intercepto, a coluna completa seria redundante (colinearidade perfeita) e os coeficientes deixariam de ser únicos.

In [ ]:
def montar_preprocessador(X, padronizar=True, remover_primeira=False):
    """Numéricas: mediana + padronização. Categóricas: moda + one-hot.
    remover_primeira=True descarta uma categoria por variável (evita a colinearidade perfeita
    com o intercepto em modelos lineares). Ajuste SÓ no treino (evita vazamento do teste)."""
    numericas = X.select_dtypes(include="number").columns.tolist()
    categoricas = [c for c in X.columns if c not in numericas]
    passos_num = [SimpleImputer(strategy="median")] + ([StandardScaler()] if padronizar else [])
    transformadores = []
    if numericas:
        transformadores.append(("num", make_pipeline(*passos_num), numericas))
    if categoricas:
        transformadores.append(("cat", make_pipeline(
            SimpleImputer(strategy="most_frequent"),
            OneHotEncoder(handle_unknown="ignore", sparse_output=False,
                          drop="first" if remover_primeira else None)), categoricas))
    return ColumnTransformer(transformadores, verbose_feature_names_out=False)


if TAMANHO_TESTE:
    X_tr_df, X_te_df, y_train, y_test = train_test_split(X_df, y, test_size=TAMANHO_TESTE, random_state=SEMENTE)
else:   # datasets minúsculos: treina com tudo; as métricas de "teste" passam a ser do próprio treino
    X_tr_df, X_te_df, y_train, y_test = X_df, X_df, y, y
    print("ATENÇÃO: TAMANHO_TESTE = 0, sem conjunto de teste; as métricas não medem generalização.")
# a padronização final também centraliza as colunas do one-hot: sem isso o gradiente descendente fica mal
# condicionado (dummies 0/1 correlacionadas com o intercepto) e converge muito devagar
prep = make_pipeline(montar_preprocessador(X_tr_df, padronizar=PADRONIZAR, remover_primeira=True),
                     *([StandardScaler()] if PADRONIZAR else []))
X_train = prep.fit_transform(X_tr_df)
X_test = prep.transform(X_te_df)
nomes = prep.get_feature_names_out()
print("treino:", X_train.shape, "| teste:", X_test.shape)

## 4. Modelo

O modelo com $n$ features é

$$f_{\mathbf{w},b}(\mathbf{x}) = \mathbf{w}\cdot\mathbf{x} + b$$

e o custo (erro quadrático médio dividido por 2, convenção que simplifica a derivada) é

$$J(\mathbf{w},b) = \frac{1}{2m} \sum_{i=0}^{m-1} \left(f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}\right)^2$$

O gradiente descendente atualiza **simultaneamente** todos os parâmetros:

$$w_j := w_j - \alpha \frac{\partial J}{\partial w_j}, \qquad b := b - \alpha \frac{\partial J}{\partial b}$$

$$\frac{\partial J}{\partial w_j} = \frac{1}{m} \sum_{i=0}^{m-1} \left(f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}\right)x_j^{(i)}, \qquad \frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=0}^{m-1} \left(f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}\right)$$

Com uma única feature, essas fórmulas são exatamente as do notebook original.

In [ ]:
def compute_cost(X, y, w, b):
    """Custo J(w, b) = (1/2m) * soma dos erros ao quadrado. X: (m, n), y: (m,), w: (n,), b: escalar."""
    m = X.shape[0]
    erros = X @ w + b - y
    return float(erros @ erros) / (2 * m)


def compute_gradient(X, y, w, b):
    """Devolve (dj_dw, dj_db): gradiente do custo em relação a w (n,) e a b (escalar)."""
    m = X.shape[0]
    erros = X @ w + b - y
    dj_dw = X.T @ erros / m
    dj_db = float(erros.sum()) / m
    return dj_dw, dj_db


def gradient_descent(X, y, w_in, b_in, alpha, num_iters, cost_function=compute_cost,
                     gradient_function=compute_gradient, imprimir=10):
    """Gradiente descendente em lote (todos os exemplos a cada passo).
    Devolve w, b e o histórico do custo, que deve cair a cada iteração."""
    w, b = np.array(w_in, dtype=float).copy(), float(b_in)   # não altera os valores de entrada
    J_history = []
    passo_impressao = max(1, int(np.ceil(num_iters / imprimir)))
    for i in range(num_iters):
        dj_dw, dj_db = gradient_function(X, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        J_history.append(cost_function(X, y, w, b))
        if i % passo_impressao == 0 or i == num_iters - 1:
            print(f"Iteração {i:5d}: custo {J_history[-1]:12.4f}")
        if not np.isfinite(J_history[-1]):
            raise FloatingPointError("O custo divergiu: reduza TAXA_APRENDIZADO ou padronize as features.")
    return w, b, J_history


def predict(X, w, b):
    return X @ w + b

### 4.1 Verificação do gradiente
Substitui os testes do autograder: compara o gradiente analítico com diferenças finitas num problema pequeno e aleatório. Se a implementação estiver errada, a célula falha.

In [ ]:
def gradiente_numerico(X, y, w, b, h=1e-6):
    gw = np.zeros_like(w)
    for j in range(len(w)):
        e = np.zeros_like(w); e[j] = h
        gw[j] = (compute_cost(X, y, w + e, b) - compute_cost(X, y, w - e, b)) / (2 * h)
    gb = (compute_cost(X, y, w, b + h) - compute_cost(X, y, w, b - h)) / (2 * h)
    return gw, gb

rng = np.random.default_rng(0)
Xp, yp, wp, bp = rng.normal(size=(8, 3)), rng.normal(size=8), rng.normal(size=3), 0.3
ga, gb_a = compute_gradient(Xp, yp, wp, bp)
gn, gb_n = gradiente_numerico(Xp, yp, wp, bp)
assert np.allclose(ga, gn, atol=1e-6) and np.isclose(gb_a, gb_n, atol=1e-6), "gradiente incorreto"
print("gradiente verificado: analítico = numérico")

## 5. Treinamento

In [ ]:
w_inicial = np.zeros(X_train.shape[1])
b_inicial = 0.0
print(f"custo inicial: {compute_cost(X_train, y_train, w_inicial, b_inicial):.4f}")

w, b, J_hist = gradient_descent(X_train, y_train, w_inicial, b_inicial, TAXA_APRENDIZADO, NUM_ITERACOES)
print("\nb =", round(b, 4))
display(pd.Series(w, index=nomes, name="w").round(4))

In [ ]:
# O custo deve cair a cada passo e estabilizar; se oscilar ou subir, reduza TAXA_APRENDIZADO.
plt.figure(figsize=(6, 3.5))
plt.plot(J_hist)
plt.yscale("log")
plt.xlabel("iteração")
plt.ylabel("custo J (escala log)")
plt.title("Convergência do gradiente descendente")
plt.show()
if np.any(np.diff(J_hist) > 1e-9 * max(1.0, J_hist[0])):
    print("ATENÇÃO: o custo subiu em algum passo; reduza TAXA_APRENDIZADO.")

## 6. Avaliação
Métricas do cartão: MAE, RMSE e R², sempre no **teste** (dados que o modelo não viu). Comparar com o treino diagnostica sobreajuste.

In [ ]:
def metricas_regressao(y_real, y_prev):
    return {"MAE": mean_absolute_error(y_real, y_prev),
            "RMSE": root_mean_squared_error(y_real, y_prev),
            "R2": r2_score(y_real, y_prev)}

pred_train, pred_test = predict(X_train, w, b), predict(X_test, w, b)
resultado = pd.DataFrame({"treino": metricas_regressao(y_train, pred_train),
                          "teste": metricas_regressao(y_test, pred_test)}).T
display(resultado.round(4))

In [ ]:
# Análise de resíduos: devem ficar em torno de zero, sem padrão (homocedasticidade)
# e com distribuição aproximadamente simétrica.
residuos = y_test - pred_test
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
ax[0].scatter(y_test, pred_test, s=12, alpha=0.7)
lim = [min(y_test.min(), pred_test.min()), max(y_test.max(), pred_test.max())]
ax[0].plot(lim, lim, "--", lw=1)
ax[0].set(xlabel="real", ylabel="previsto", title="Previsto × real (teste)")
ax[1].scatter(pred_test, residuos, s=12, alpha=0.7)
ax[1].axhline(0, ls="--", lw=1)
ax[1].set(xlabel="previsto", ylabel="resíduo", title="Resíduos × previsto")
ax[2].hist(residuos, bins=30)
ax[2].set(xlabel="resíduo", title="Distribuição dos resíduos")
plt.tight_layout()
plt.show()

In [ ]:
# Com uma única feature, reproduz o gráfico do original: dados e reta ajustada.
if X_train.shape[1] == 1:
    ordem = np.argsort(X_train[:, 0])
    plt.scatter(X_train[:, 0], y_train, marker="x", label="treino")
    plt.plot(X_train[ordem, 0], pred_train[ordem], label="reta ajustada")
    plt.xlabel(nomes[0] + (" (padronizada)" if PADRONIZAR else ""))
    plt.ylabel(COLUNA_ALVO)
    plt.legend()
    plt.show()
else:
    print("Mais de uma feature: o gráfico da reta não se aplica; use os gráficos de resíduos acima.")

## 7. Validação com a solução de referência
O cartão do sistema descreve a regressão linear pela **forma fechada** (equações normais, `LinearRegression`). Com convergência suficiente, o gradiente descendente precisa chegar aos mesmos coeficientes. Se a diferença for grande, aumente `NUM_ITERACOES` ou ajuste `TAXA_APRENDIZADO`.

In [ ]:
ref = LinearRegression().fit(X_train, y_train)
comparacao = pd.DataFrame({"gradiente descendente": np.r_[w, b], "LinearRegression": np.r_[ref.coef_, ref.intercept_]},
                          index=list(nomes) + ["intercepto (b)"])
comparacao["diferença"] = comparacao.iloc[:, 0] - comparacao.iloc[:, 1]
display(comparacao.round(5))
if comparacao["diferença"].abs().max() > 1e-2 * max(1.0, comparacao.iloc[:, 1].abs().max()):
    print("ATENÇÃO: o gradiente descendente ainda não chegou ao ótimo; aumente NUM_ITERACOES ou TAXA_APRENDIZADO.")
print(f"RMSE teste · GD: {root_mean_squared_error(y_test, pred_test):.4f} | "
      f"LinearRegression: {root_mean_squared_error(y_test, ref.predict(X_test)):.4f}")

## 8. Previsão para novos dados
Novos registros passam **pelo mesmo pré-processamento** ajustado no treino. Troque `novos` pelos seus casos (mesmas colunas de `features`); no original, eram as cidades candidatas de 35 mil e 70 mil habitantes.

In [ ]:
novos = X_df.sample(3, random_state=SEMENTE)          # >>> ADAPTAR: DataFrame com as mesmas colunas de `features`
previsoes = predict(prep.transform(novos), w, b)
display(novos.assign(previsao=previsoes))

## 9. Próximos passos (segundo o cartão do sistema)
- **Pressupostos:** relação linear, erros independentes e homocedásticos, pouca multicolinearidade. Os gráficos de resíduos da seção 6 são a primeira checagem.
- **Prefira Ridge ou Lasso** se houver multicolinearidade ou excesso de features; **boosting** se os resíduos mostrarem estrutura não linear.
- Alvo muito assimétrico: avalie transformar com `log` antes de ajustar.